# Classification avec SIFT + SVM

## Objectif
Utiliser SIFT (Scale-Invariant Feature Transform) pour extraire les points clés (keypoints)
et les descripteurs, puis classifier avec SVM.

Pipeline :
Images → SIFT → Descripteurs → SVM → Prédiction

## Etape 1  : Importation

In [22]:
import cv2
import os
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

## Etape 2 : Chargement des images 

In [23]:
def load_images(path, label):
    images = []
    labels = []
    
    for img_name in os.listdir(path):
        img_path = os.path.join(path, img_name)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
        
        img = cv2.resize(img, (128, 128))
        images.append(img)
        labels.append(label)
    
    return images, labels

cats, cats_labels = load_images("C:/Users/4B/image_project/dataset/cats", 0)
dogs, dogs_labels = load_images("C:/Users/4B/image_project/dataset/dogs", 1)

X = cats + dogs
y = cats_labels + dogs_labels

print("Nombre d’images :", len(X))

Nombre d’images : 482


## nombres des images 

In [24]:
print("Cats:", len(cats))
print("Dogs:", len(dogs))
print("Total:", len(X))

Cats: 241
Dogs: 241
Total: 482


## Etape 3 : Création et Extraction SIFT

In [25]:
sift = cv2.SIFT_create()

sift_features = []

for img in X:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    
    # إذا ما لقا حتى descriptor
    if descriptors is None:
        descriptors = np.zeros((1, 128))
    
    # ناخدو المتوسط باش يكون vector واحد
    feature = np.mean(descriptors, axis=0)
    
    sift_features.append(feature)
    
print("Nombre de vecteurs :", len(sift_features))
print("Feature size:", len(sift_features[0]))

Nombre de vecteurs : 482
Feature size: 128


## Etape 4 : Préparation des données

In [26]:
X_data = np.array(sift_features)
y_data = np.array(y)

print(X_data.shape)
print(y_data.shape)

(482, 128)
(482,)


## Etape 5 : Normalisation

In [27]:
scaler = StandardScaler()
X_data = scaler.fit_transform(X_data)

## Etape 6 : Division des données

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)
print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 385
Test: 97


##  Etape 7 :  SVM Training

In [29]:
model = SVC(kernel='rbf')
model.fit(X_train, y_train)
print("Modèle entraîné ✔️")

Modèle entraîné ✔️


## Etape 8 : Prédiction 

In [30]:
y_pred = model.predict(X_test)

## Etape 9 : Évaluation

In [31]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.5979381443298969
              precision    recall  f1-score   support

           0       0.62      0.58      0.60        50
           1       0.58      0.62      0.60        47

    accuracy                           0.60        97
   macro avg       0.60      0.60      0.60        97
weighted avg       0.60      0.60      0.60        97



## Conclusion

SIFT permet d’extraire des caractéristiques robustes basées sur les points clés.
Les résultats seront comparés avec HOG et LBP.